# Chart Explanation — LLM + RAG

Google Colab prototype for explaining WealthWise charts using retrieved financial context.

**Flow:** Chart data → RAG retrieval → chart-aware prompt → Ollama LLM → explanation.

In [1]:
!pip -q install chromadb sentence-transformers pandas numpy requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:
import requests
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer


## 1. Example chart data

Replace this with chart data from the WealthWise Django backend.

In [3]:
chart_data = pd.DataFrame({
    'category': ['Food', 'Shopping', 'Transport', 'Entertainment', 'Groceries'],
    'spending': [4200, 3798, 800, 649, 1850],
    'budget': [5000, 4000, 3000, 2000, 5000]
})
display(chart_data)


,category,spending,budget
0,Food,4200,5000
1,Shopping,3798,4000
2,Transport,800,3000
3,Entertainment,649,2000
4,Groceries,1850,5000


## 2. Create RAG knowledge base

In [4]:
documents = [
    'Food monthly budget is INR 5000 and current spending is INR 4200.',
    'Shopping monthly budget is INR 4000 and current spending is INR 3798.',
    'Transport monthly budget is INR 3000 and current spending is INR 800.',
    'Entertainment monthly budget is INR 2000 and current spending is INR 649.',
    'Groceries monthly budget is INR 5000 and current spending is INR 1850.',
    'Explain financial charts only from available WealthWise data.'
]

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(documents, normalize_embeddings=True).tolist()

client = chromadb.Client()
collection = client.get_or_create_collection('wealthwise_chart_context')
collection.upsert(ids=[f'doc_{i}' for i in range(len(documents))], documents=documents, embeddings=embeddings)
print('Documents stored:', collection.count())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documents stored: 6


## 3. Retrieve relevant context

In [5]:
def retrieve_context(query, top_k=5):
    q = embedding_model.encode([query], normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=q, n_results=top_k)
    return result['documents'][0]

context = retrieve_context('Explain spending by category and identify important budget patterns.')
for item in context:
    print('-', item)


- Groceries monthly budget is INR 5000 and current spending is INR 1850.
- Food monthly budget is INR 5000 and current spending is INR 4200.
- Transport monthly budget is INR 3000 and current spending is INR 800.
- Entertainment monthly budget is INR 2000 and current spending is INR 649.
- Shopping monthly budget is INR 4000 and current spending is INR 3798.


## 4. Build chart-aware prompt

In [6]:
def build_prompt(chart, context):
    return f'''You are WealthWise, a financial chart explanation assistant.

Explain the supplied chart clearly for a normal user.
Use ONLY the chart values and retrieved context. Never invent data.
Identify the most important comparison, trend, unusual value, or budget risk.
Mention exact values when useful.
Return one short summary paragraph followed by exactly 3 key insights.
Do not provide unsupported financial advice.

CHART DATA:
{chart.to_json(orient='records')}

RETRIEVED CONTEXT:
{chr(10).join('- '+x for x in context)}
'''

prompt = build_prompt(chart_data, context)
print(prompt)


You are WealthWise, a financial chart explanation assistant.

Explain the supplied chart clearly for a normal user.
Use ONLY the chart values and retrieved context. Never invent data.
Identify the most important comparison, trend, unusual value, or budget risk.
Mention exact values when useful.
Return one short summary paragraph followed by exactly 3 key insights.
Do not provide unsupported financial advice.

CHART DATA:
[{"category":"Food","spending":4200,"budget":5000},{"category":"Shopping","spending":3798,"budget":4000},{"category":"Transport","spending":800,"budget":3000},{"category":"Entertainment","spending":649,"budget":2000},{"category":"Groceries","spending":1850,"budget":5000}]

RETRIEVED CONTEXT:
- Groceries monthly budget is INR 5000 and current spending is INR 1850.
- Food monthly budget is INR 5000 and current spending is INR 4200.
- Transport monthly budget is INR 3000 and current spending is INR 800.
- Entertainment monthly budget is INR 2000 and current spending is IN

## 5. Generate explanation with Ollama

For local execution use `http://localhost:11434`. For Colab, replace it with your Cloudflare Tunnel URL.

In [7]:
OLLAMA_URL = 'https://bikini-chosen-bye-collective.trycloudflare.com/api/generate'
OLLAMA_MODEL = 'llama3.2'

def generate_explanation(prompt):
    r = requests.post(OLLAMA_URL, json={'model': OLLAMA_MODEL, 'prompt': prompt, 'stream': False}, timeout=120)
    r.raise_for_status()
    return r.json()['response']

try:
    explanation = generate_explanation(prompt)
    print(explanation)
except Exception as e:
    print('Ollama unavailable:', e)


**Summary:** The chart shows the current monthly expenditure of an individual across five categories: Food, Shopping, Transport, Entertainment, and Groceries. The data highlights areas where expenses exceed budgets, indicating potential budget risks.

**Key Insights:**

1. **Budget risk in Transportation**: The current spending in the Transport category (INR 800) is INR 2200 lower than its monthly budget of INR 3000, making it a notable risk area.
2. **Groceries vs. Budget**: The individual's current spending in Groceries (INR 1850) is INR 3150 below its monthly budget of INR 5000, indicating a significant saving opportunity.
3. **Shopping expenses close to budget**: The current spending in the Shopping category (INR 3798) is only INR 202 from its monthly budget of INR 4000, suggesting that this expense is relatively well-managed.


## 6. Fallback explanation

In [8]:
chart_data['utilization_pct'] = chart_data['spending'] / chart_data['budget'] * 100
highest = chart_data.loc[chart_data['spending'].idxmax()]
highest_util = chart_data.loc[chart_data['utilization_pct'].idxmax()]
fallback = f'''The chart shows that {highest.category} has the highest spending at INR {highest.spending:,.0f}.

Key insights:
1. {highest.category} is the largest spending category.
2. {highest_util.category} has the highest budget utilization at {highest_util.utilization_pct:.1f}%.
3. Categories with high budget utilization should be monitored against their remaining budget.'''
if 'explanation' not in globals(): print(fallback)


## WealthWise integration

The frontend sends chart type, labels, values, date range, and the user's question to Django. Django retrieves relevant financial facts from ChromaDB and sends the chart plus retrieved context to the LLM. The LLM explains the chart without changing its underlying data.